# Toxic Content & Hate Speech Linguistic Feature Extractor

**Domain:** Production Natural Language Processing & Feature Engineering  
**Author:** Néstor León del Campo  

---

## Technical Overview
Pipeline integral de procesamiento lingüístico en español para extracción morfosintáctica (PoS tagging), Named Entity Recognition (NER), análisis de dependencias gramaticales y vectorización híbrida TF-IDF sobre textos de redes sociales.

---


### System Objectives

Construir representaciones vectoriales densas y dispersas sobre lenguaje informal y complejo, eliminando ruido fonético y ortográfico sin pérdida de coherencia sintáctica para tareas de clasificación de toxicidad.

# Parte 1: carga y preprocesamiento del texto a analizar

Observa las diferentes librerías que se están importando.

In [1]:
import pathlib
import spacy
import pandas as pd
import io
from pathlib import Path
from spacy import displacy
import csv
import es_core_news_md

El siguiente código simplemente carga y preprocesa el texto. Para ello, lo primero que hace es cargar un modelo de lenguaje previamente entrenado. En este caso, se utiliza <i>es_core_news_md</i>: 

https://spacy.io/models/es#es_core_news_md


In [2]:
nlp = es_core_news_md.load()

El objeto <i>nlp</i> permite utilizar el modelo de lenguaje cargado, de forma que se puede procesar un texto y obtenerlo en su versión preprocesada. Así, nos permite realizar las diferentes tareas. En este caso, vamos a utilizar el pipeline para hacer un preprocesamiento básico, que consiste en tokenizar el texto.

In [3]:
"""
ESTRATEGIA DE LIMPIEZA IN-MEMORY
Refactorización para evitar I/O intermedio en disco.
1. Leemos el fichero binario y limpiamos bytes corruptos al vuelo hacia un buffer (io.BytesIO).
2. Cargamos el DataFrame directamente desde ese buffer.
3. Aplicamos la lógica de corrección de codificación (mojibake).
"""
# Configuración de rutas
input_path = Path("./02Dataset_anonimizado.csv")
output_path = Path("comentarios_limpio_utf8.csv")

# Bytes problemáticos (garbage characters)
BAD_BYTES = {0xBF, 0xA1, 0xB3}

# Diccionario de correcciones manuales para patrones corruptos recurrentes
reemplazos_manuales = {
    'í§': 'ç',   # barí§a -> barça
    'í¼': 'ü',   # vergí¼enza -> vergüenza
    'í±': 'ñ',   # espaí±a -> españa
    'í¡': 'á',
    'í©': 'é',
    'í³': 'ó',
    'íº': 'ú',
    'í\xad': 'í', 
    'Ã±': 'ñ',    
    'Ã³': 'ó'
}

def clean_bytes_in_memory(file_path):
    """
    Lee el fichero en modo binario, filtra los bytes malos y devuelve
    un objeto BytesIO que pandas puede leer como si fuera un archivo.
    """
    with open(file_path, "rb") as f:
        # Leemos todo el contenido (si el dataset es masivo >GBs, habría que hacerlo por chunks)
        raw_content = f.read()
    
    # Filtrado de bytes usando una list comprehension generadora para eficiencia
    cleaned_content = bytes(b for b in raw_content if b not in BAD_BYTES)
    
    return io.BytesIO(cleaned_content)

def arreglar_mojibake_estandar(s):
    """Intenta arreglar el típico error de UTF-8 leído como Latin-1."""
    if not isinstance(s, str):
        return s
    try:
        return s.encode("latin1").decode("utf-8")
    except UnicodeError:
        return s

def limpieza_completa(text):
    """Pipeline de limpieza de texto: Mojibake -> Manual."""
    if not isinstance(text, str):
        return text
    
    # 1. Arreglo estándar de codificación
    text = arreglar_mojibake_estandar(text)
    
    # 2. Parches manuales específicos
    for sucio, limpio in reemplazos_manuales.items():
        text = text.replace(sucio, limpio)
        
    return text

# --- EJECUCIÓN ---

print("1. Cargando y limpiando bytes en memoria...")
# Usamos el buffer en memoria en lugar del archivo intermedio
buffer_limpio = clean_bytes_in_memory(input_path)

print("2. Creando DataFrame...")
# Leemos desde el buffer usando latin1 como base segura
df = pd.read_csv(
    buffer_limpio,
    sep=";",
    encoding="latin1", 
    low_memory=False
)

print(f"   Dimensiones iniciales: {df.shape}")

print("3. Aplicando correcciones de texto (Mojibake + Manual)...")
# Detectamos columnas de tipo objeto (string) automáticamente
text_cols = df.select_dtypes(include="object").columns

for col in text_cols:
    df[col] = df[col].apply(limpieza_completa)

# 4. Guardado final
print(f"4. Guardando resultado en {output_path}...")
df.to_csv(output_path, sep=";", index=False, encoding="utf-8")

print("¡Proceso terminado!")
print("Prueba de control:", limpieza_completa("el barí§a es una vergí¼enza"))

1. Cargando y limpiando bytes en memoria...
2. Creando DataFrame...
   Dimensiones iniciales: (575029, 16)
3. Aplicando correcciones de texto (Mojibake + Manual)...
4. Guardando resultado en comentarios_limpio_utf8.csv...
¡Proceso terminado!
Prueba de control: el barça es una vergüenza


In [4]:
"""
Cargamos solo las columnas que necesitamos del dataset limpio en UTF-8,limitándonos a las primeras 10.000 filas para agilizar el proceso.
Creamos una nueva columna "ODIO_BINARIO" que marca 1 si el mensaje contiene odio (INTENSIDAD > 0) y 0 si no lo contiene.
Luego mostramos cuántos mensajes hay de cada tipo.
"""
filename = "comentarios_limpio_utf8.csv"  # ruta del dataset
lines_number = 10000                       # número de filas a cargar
columnas_a_usar = ["CONTENIDO A ANALIZAR", "INTENSIDAD"]  # columnas relevantes

# Carga del CSV
data = pd.read_csv( filename, nrows=lines_number, sep=";", encoding="utf-8", usecols=columnas_a_usar)

# Creación de la columna binaria de odio
data["ODIO_BINARIO"] = (data["INTENSIDAD"] > 0).astype(int)

# Conteo rápido de mensajes con/sin odio
print(data["ODIO_BINARIO"].value_counts())

ODIO_BINARIO
0    9252
1     748
Name: count, dtype: int64


El código anterior carga el archivo CSV (opcionalmente con un límite de líneas a leer) y genera la variable <i>data</i>, que contiene un Dataframe (https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) con los datos leídos del CSV.

Te vendrá bien conocer la siguiente documentación:
<ul>
    <li>https://spacy.io/api/doc</li>
    <li>https://spacy.io/api/token</li>
    <li>https://spacy.io/api/morphology#morphanalysis</li>
</ul>

### Playground

Utiliza este espacio para hacer pruebas y ensayos con las variables generadas con el código previo. A modo de ejemplo, se ofrece código que realiza las siguientes tareas: 


- leer un número dado de líneas del Dataframe y generar dos listas con los valores (se pueden leer directamente del DataFrame, se muestra el ejemplo como una opción más)
- procesar el texto de cada comentario


Para procesarlo, hay utilizar el objeto <i>nlp</i> y así obtener objetos de la clase <i>Doc</i> (https://spacy.io/api/doc)

Visita la documentación de dicha clase y experimenta probando las diferentes funciones y atributos 

In [5]:
"""
En lugar de recorrer el DataFrame fila por fila y llamar a 'nlp()' repetidamente (lo que sería muy lento),
utilizamos el método 'nlp.pipe()'.

Este método procesa los textos en lotes (streaming), lo que es mucho más eficiente computacionalmente.
El resultado se almacena en una nueva columna llamada "DOC". De esta forma, en las preguntas posteriores
(contar palabras, oraciones, entidades, etc.) solo tendremos que consultar esta columna ya procesada,
sin necesidad de volver a tokenizar el texto.
"""

# 1. Convertimos la columna de texto a una lista de strings (asegurando que sean strings)
textos = data["CONTENIDO A ANALIZAR"].astype(str).tolist()

print(f"Iniciando el procesamiento masivo de {len(textos)} registros con nlp.pipe()...")

# 2. Usamos nlp.pipe para procesar todo el lote
docs = list(nlp.pipe(textos))

# 3. Guardamos los objetos procesados en el DataFrame
data["DOC"] = docs

print("¡Procesamiento completado!")
print("Ejemplo del primer doc procesado:", data["DOC"][0])

Iniciando el procesamiento masivo de 10000 registros con nlp.pipe()...
¡Procesamiento completado!
Ejemplo del primer doc procesado: el barça nunca acaeza ante un segundo b ni ante un tercera , ya estan los arbitros para impedirlo....lo de messi es una autentica vergüenza


<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Pregunta 1.</span>
<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">¿Cuántos registros contiene el corpus?</span>

In [6]:
# Usamos la función len() para contar las filas del DataFrame 'data'
total_registros = len(data)
print(f"El corpus contiene {total_registros} registros.")

El corpus contiene 10000 registros.


<b>Incluye aquí, debajo de la línea, la explicación de tu respuesta</b>
<hr>
 

Para determinar el número de registros del corpus, se ha utilizado la función nativa de Python len() aplicada sobre el objeto DataFrame data. Esta función devuelve la longitud del objeto, lo que corresponde al número total de filas (observaciones) cargadas. Dado que en la etapa de carga establecimos la variable lines_number en 10.000, este valor refleja esa restricción.

<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Pregunta 2.</span>
<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">¿Cuántas palabras totales hay en los comentarios del corpus?</span>

In [7]:
"""
CÁLCULO DE PALABRAS TOTALES
Para contar palabras reales y no solo tokens técnicos, iteramos sobre cada documento
y filtramos usando .is_alpha. Esto descarta puntuación (.,!) y números, quedándonos
solo con palabras formadas por letras.
"""
# 1. Calcular el número TOTAL (esto devuelve un entero)
total_palabras = data["DOC"].apply(lambda doc: len([token for token in doc if token.is_alpha])).sum()

print(f"Hay un total de {total_palabras} palabras reales en los comentarios del corpus.")

# 2. Para ver las palabras (el texto), extraemos las del primer comentario
palabras_primer_doc = [token.text for token in data["DOC"][0] if token.is_alpha]

print("Las palabras del primer comentario:")
print(palabras_primer_doc)

Hay un total de 208271 palabras reales en los comentarios del corpus.
Las palabras del primer comentario:
['el', 'barça', 'nunca', 'acaeza', 'ante', 'un', 'segundo', 'b', 'ni', 'ante', 'un', 'tercera', 'ya', 'estan', 'los', 'arbitros', 'para', 'impedirlo', 'lo', 'de', 'messi', 'es', 'una', 'autentica', 'vergüenza']


<b>Incluye aquí, debajo de la línea, la explicación de tu respuesta</b>
<hr>
 

Para obtener el número real de palabras, se ha refinado el método de conteo. A diferencia de la función básica len(doc), que contabiliza todos los tokens (incluyendo signos de puntuación y símbolos), en este caso hemos aplicado un filtro para seleccionar únicamente los tokens alfabéticos (utilizando el atributo .is_alpha de spaCy). De esta forma, excluimos del recuento las comas, puntos y otros signos ortográficos, obteniendo una suma total que refleja fielmente el volumen léxico del corpus que consta de 208271 palabras.

<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Pregunta 3.</span>
<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">¿Cuál el número promedio de palabras en cada comentario?</span>

In [8]:
promedio_palabras= total_palabras/total_registros

print(f" El número promedio de palabras por comentario es de {promedio_palabras:.2f} palabras")

 El número promedio de palabras por comentario es de 20.83 palabras


<b>Incluye aquí, debajo de la línea, la explicación de tu respuesta</b>
<hr>
 

El cálculo se ha realizado mediante una media aritmética simple: se ha dividido el sumatorio total de palabras del corpus (resultado de la Pregunta 2) entre el número total de registros (resultado de la Pregunta 1). El número promedio de palabras por comentario es de 20.83 palabras.

<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Pregunta 4.</span>
<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Considerando dos grupos de comentarios (odio y no odio) ¿Cuál el número promedio de palabras en los comentarios de cada grupo?</span>

In [9]:
# 1. Calculamos el número de palabras para cada fila
data['NUM_PALABRAS'] = data['DOC'].apply(lambda doc: len([token for token in doc if token.is_alpha]))

# 2. Agrupamos por la etiqueta de odio y sacamos la media
promedio_por_grupo = data.groupby("ODIO_BINARIO")["NUM_PALABRAS"].mean()

print("Promedio de palabras por grupo:")
print(promedio_por_grupo)

Promedio de palabras por grupo:
ODIO_BINARIO
0    20.733139
1    21.989305
Name: NUM_PALABRAS, dtype: float64


<b>Incluye aquí, debajo de la línea, la explicación de tu respuesta</b>
<hr>
 

Se ha calculado primero la cantidad de palabras por fila y se ha almacenado en la columna auxiliar NUM_PALABRAS. Posteriormente, mediante la función .groupby("ODIO_BINARIO") y .mean(), hemos obtenido el promedio de palabras para cada clase.

**Resultado:** Los mensajes de odio presentan una media de 21.98 palabras, ligeramente superior a las 20.73 palabras de los mensajes sin odio, indicando que en este corpus el contenido ofensivo tiende a ser un poco más extenso.

<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Pregunta 5.</span>
<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Considerando dos grupos de comentarios (odio y no odio) ¿Cuál es el número promedio de oraciones en los comentarios de cada grupo?</span>

In [10]:
#1. Calculamos el número de oraciones para cada documento
data['NUM_ORACIONES'] = data['DOC'].apply(lambda doc: len(list(doc.sents)))

#2. Agrupamos por 'ODIO_BINARIO' y calculamos la media
promedio_oraciones = data.groupby("ODIO_BINARIO")["NUM_ORACIONES"].mean()

print("Promedio de oraciones por grupo:")
print(promedio_oraciones)

Promedio de oraciones por grupo:
ODIO_BINARIO
0    1.781128
1    1.770053
Name: NUM_ORACIONES, dtype: float64


<b>Incluye aquí, debajo de la línea, la explicación de tu respuesta</b>
<hr>
 

Para esta métrica, utilizamos el atributo .sents de spaCy, que realiza la segmentación automática de oraciones. Contamos las frases de cada comentario y calculamos el promedio para cada grupo.

**Resultado:** Los datos muestran una media casi idéntica (1.78 para no odio vs 1.77 para odio). Esto indica que ambos tipos de mensajes suelen ser breves y de estructura similar.

<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Pregunta 6.</span>
<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Considerando dos grupos de comentarios (odio y no odio) ¿Cuál es el porcentaje de comentarios que contienen entidades NER en cada grupo?</span>

In [11]:
# 1. Creamos una columna binaria: 1 si el doc tiene alguna entidad (doc.ents), 0 si no.
data['TIENE_ENTIDADES'] = data['DOC'].apply(lambda doc: len(doc.ents) > 0).astype(int)

# 2. Agrupamos por ODIO_BINARIO y calculamos la media.
porcentaje_entidades = data.groupby("ODIO_BINARIO")["TIENE_ENTIDADES"].mean() * 100

print("Porcentaje de comentarios con entidades NER por grupo:")
print(porcentaje_entidades)

Porcentaje de comentarios con entidades NER por grupo:
ODIO_BINARIO
0    32.857760
1    37.433155
Name: TIENE_ENTIDADES, dtype: float64


<b>Incluye aquí, debajo de la línea, la explicación de tu respuesta</b>
<hr>
 

Para calcular esta métrica, utilizamos el atributo .ents de spaCy para detectar si un mensaje menciona entidades reales (personas, lugares, organizaciones). Creamos una variable binaria y calculamos el porcentaje promedio para cada clase.

**Resultado:** Los mensajes de odio contienen entidades con mayor frecuencia (37.43%) que los mensajes sin odio (32.85%). Esto sugiere que el discurso de odio tiende a ser más específico y dirigido contra objetivos concretos, en lugar de ser abstracto.

<span style="font-size: 16pt; font-weight: bold; color: #0098cd;">Plantea tus propias preguntas</span>

<span><b>Plantea al menos 4 características</b> del texto cuyo análisis permita una caracterización completa del texto. Puedes utilizar recomendaciones proporcionadas por la IA Generativa, si así lo deseas. Para cada una de las características planteadas, obtén valores separados para los grupos ODIO/NO-ODIO.</span>

<span>En la explicación aportada, deberás <b>explicar el significado de la característica planteada</b> así como la importancia de ésta en la caracterización del texto.</span>

<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Característica adicional 1.</span>


<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Ratio de Mayúsculas</span>

In [12]:
def calcular_ratio_gritos(doc):
    # Filtramos solo tokens que sean palabras (alfabéticos)
    palabras = [token for token in doc if token.is_alpha]
    if len(palabras) == 0:
        return 0.0
    # Contamos palabras que están TODAS en mayúsculas
    n_mayusculas = sum(1 for token in palabras if token.is_upper)
    return n_mayusculas / len(palabras)

data['RATIO_MAYUSCULAS'] = data['DOC'].apply(calcular_ratio_gritos)

print("Promedio de palabras en mayúsculas por grupo (0=No Odio, 1=Odio):")
print(data.groupby("ODIO_BINARIO")["RATIO_MAYUSCULAS"].mean())

Promedio de palabras en mayúsculas por grupo (0=No Odio, 1=Odio):
ODIO_BINARIO
0    0.000216
1    0.000326
Name: RATIO_MAYUSCULAS, dtype: float64


<b>Incluye aquí, debajo de la línea, la explicación de la característica propuesta y su motivación. Incluye también una explicación del código fuente aportado.</b>
<hr>
 

**Explicación y motivación:** Se mide la proporción de palabras escritas totalmente en mayúsculas. En redes sociales, el uso sostenido de mayúsculas denota agresividad o ira (gritar). Los resultados obtenidos: los mensajes de odio utilizan este recurso con una frecuencia un 50% superior (0.00032) a los mensajes neutros (0.00021).

**Explicación del código:** La función calcular_ratio_gritos filtra primero el texto para analizar solo palabras reales (is_alpha), ignorando signos de puntuación. Posteriormente, cuenta cuántas de esas palabras tienen la propiedad is_upper (todo mayúsculas) y divide esa cantidad entre el total de palabras para obtener el ratio.

<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Característica adicional 2.</span>


<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Complejidad léxica</span>

In [13]:
# Definimos función para calcular la longitud media de las palabras
def longitud_media_palabras(doc):
    # Filtramos para quedarnos solo con palabras (sin puntuación)
    palabras = [token for token in doc if token.is_alpha]
    
    if len(palabras) == 0:
        return 0.0
    
    # Sumamos la longitud de cada palabra y dividimos por el número total de palabras
    total_caracteres = sum(len(token.text) for token in palabras)
    return total_caracteres / len(palabras)

# Aplicamos la función
data['LONGITUD_MEDIA_PALABRA'] = data['DOC'].apply(longitud_media_palabras)

# Mostramos resultados
print("Longitud media de palabras por grupo:")
print(data.groupby("ODIO_BINARIO")["LONGITUD_MEDIA_PALABRA"].mean())

Longitud media de palabras por grupo:
ODIO_BINARIO
0    4.693930
1    4.825151
Name: LONGITUD_MEDIA_PALABRA, dtype: float64


<b>Incluye aquí, debajo de la línea, la explicación de la característica propuesta y su motivación. Incluye también una explicación del código fuente aportado.</b>
<hr>
 

**Explicación y motivación:** He elegido esta métrica para distinguir si el odio es "impulsivo" o si está más "pensado". La lógica es sencilla: el insulto rápido suele usar palabras cortas (tipo "rata", "tonto"), mientras que el odio político o ideológico usa palabras más largas para justificarse (tipo "delincuencia", "traidores"). El resultado (4.82 en odio frente a 4.69) nos dice que en este dataset los haters no solo insultan, sino que usan un lenguaje más complejo y vocabulario específico para atacar.

**Explicación del código:** El código limpia el texto para quedarse solo con las palabras reales (quitando comas y puntos). Después, simplemente cuenta las letras de todas esas palabras y las divide entre el total para sacar la media.

<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Característica adicional 3.</span>


<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Grado de subjetividad mediante adjetivos</span>

In [14]:
# Definimos función para calcular el porcentaje de adjetivos
def porcentaje_adjetivos(doc):
    if len(doc) == 0:
        return 0.0
    # Usamos el atributo .pos_ (Part Of Speech) para contar los Adjetivos (ADJ)
    n_adjetivos = sum(1 for token in doc if token.pos_ == "ADJ")
    
    # Devolvemos el porcentaje sobre el total de tokens
    return (n_adjetivos / len(doc)) * 100

# Aplicamos la función
data['PORCENTAJE_ADJETIVOS'] = data['DOC'].apply(porcentaje_adjetivos)

# Mostramos resultados
print("Porcentaje promedio de adjetivos por grupo:")
print(data.groupby("ODIO_BINARIO")["PORCENTAJE_ADJETIVOS"].mean())

Porcentaje promedio de adjetivos por grupo:
ODIO_BINARIO
0    6.442297
1    7.819131
Name: PORCENTAJE_ADJETIVOS, dtype: float64


<b>Incluye aquí, debajo de la línea, la explicación de la característica propuesta y su motivación. Incluye también una explicación del código fuente aportado.</b>
<hr>
 

**Explicación y motivación:** He elegido esta métrica para medir qué tan subjetivo es el texto. La lógica es sencilla: cuando narras un hecho (objetivo), usas sustantivos y verbos. Pero cuando insultas o atacas a alguien, necesitas adjetivos para calificarlo negativamente ("corrupto", "falso", "inútil"). Los resultados (7.82% en mensajes de odio frente a 6.44% en neutros) confirman la teoría: los mensajes de odio tienen una carga de adjetivos mucho mayor porque se basan en emitir juicios de valor y opiniones negativas, en lugar de dar información objetiva.

**Explicación del código:** Usamos la propiedad .pos_ (Part-Of-Speech) de spaCy, que nos dice la categoría gramatical de cada palabra. El código simplemente cuenta cuántas palabras tienen la etiqueta ADJ (Adjetivo) y calcula qué porcentaje representan del total del mensaje.

<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Característica adicional 4.</span>


<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Intensidad expresiva mediante el uso de exclamaciones</span>

In [15]:
# Definimos función para contar signos de exclamación
def cuenta_exclamaciones(doc):
    return sum(1 for token in doc if token.text in ["!", "¡"])

# Aplicamos la función
data['NUM_EXCLAMACIONES'] = data['DOC'].apply(cuenta_exclamaciones)

# Mostramos resultados
print("Promedio de signos de exclamación por grupo:")
print(data.groupby("ODIO_BINARIO")["NUM_EXCLAMACIONES"].mean())

Promedio de signos de exclamación por grupo:
ODIO_BINARIO
0    0.243947
1    0.339572
Name: NUM_EXCLAMACIONES, dtype: float64


<b>Incluye aquí, debajo de la línea, la explicación de la característica propuesta y su motivación. Incluye también una explicación del código fuente aportado.</b>
<hr>
 

**Explicación y motivación:** Esta característica busca cuantificar la "intensidad emocional" o el volumen de la conversación. La premisa es que el discurso de odio, al ser visceral, tiende a abusar de los signos de exclamación para imponer su mensaje o denotar indignación. Los resultados avalan esta teoría: los mensajes de odio presentan un promedio de 0.34 signos de exclamación, frente a los 0.24 de los mensajes neutros. Este incremento del 40% indica claramente que el hater escribe con mucha más "agitación" ortográfica.

**Explicación del código:** La función cuenta_exclamaciones recorre el texto token a token y suma 1 cada vez que encuentra un signo de apertura (¡) o de cierre (!), devolviendo el total acumulado para cada comentario.

<span style="font-size: 16pt; font-weight: bold; color: #0098cd;">Reflexión final</span>

<span>Una de las utilidades de la caracterización de texto es que nos sirve como etapa de <i>feature-extraction</i> (extración de características) de cara a un posterior sistema de clasificación. Es pertinente, por tanto, reflexionar sobre la capacidad discriminatoria de cada una de las características extraídas. </span>

<span> Responde, para ello, a la siguiente pregunta.</span>

<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Reflexión final.</span>
<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">¿Es posible utilizar alguna de las características extraídas en las preguntas anteriores para determinar si un mensaje contiene odio? Justifica tu respuesta con el análisis estadístico que consideres necesario.</span>

In [16]:
"""
ANÁLISIS DE CORRELACIÓN
Para determinar la capacidad discriminatoria de las características, calculamos la correlación de Pearson
entre cada una de las variables generadas y la variable objetivo 'ODIO_BINARIO'.
Esto nos dará un valor entre -1 y 1 que indica qué tan fuerte es la relación lineal.
"""

# Seleccionamos las columnas numéricas que hemos creado
features = [
    "ODIO_BINARIO",
    "NUM_PALABRAS",
    "NUM_ORACIONES",
    "TIENE_ENTIDADES",
    "RATIO_MAYUSCULAS",
    "LONGITUD_MEDIA_PALABRA",
    "PORCENTAJE_ADJETIVOS",
    "NUM_EXCLAMACIONES"
]

# Calculamos la matriz de correlación y mostramos solo la columna de ODIO ordenadada
correlaciones = data[features].corr(numeric_only=True)["ODIO_BINARIO"].sort_values(ascending=False)

print("Correlación de cada feature con la variable objetivo (ODIO_BINARIO):")
print(correlaciones)

Correlación de cada feature con la variable objetivo (ODIO_BINARIO):
ODIO_BINARIO              1.000000
PORCENTAJE_ADJETIVOS      0.036376
NUM_EXCLAMACIONES         0.027025
TIENE_ENTIDADES           0.025559
LONGITUD_MEDIA_PALABRA    0.022024
NUM_PALABRAS              0.005077
RATIO_MAYUSCULAS          0.002020
NUM_ORACIONES            -0.001258
Name: ODIO_BINARIO, dtype: float64


<b>Incluye aquí, debajo de la línea, la explicación de tu respuesta</b>
<hr>
 

Sí, estas características aportan información valiosa para determinar si un mensaje contiene odio, aunque su poder predictivo individual es limitado y funcionan mejor combinadas. Basándonos en la matriz de correlación de Pearson generada, podemos extraer las siguientes conclusiones para el diseño de un clasificador:

1. Los mejores predictores (Top Features):
   * Subjetividad (PORCENTAJE_ADJETIVOS  0.036): Es la variable con mayor correlación positiva. Esto confirma estadísticamente que el rasgo más distintivo del odio en este corpus es el uso de lenguaje calificativo y juicios de valor.
   * Intensidad (NUM_EXCLAMACIONES  0.027): Se confirma como el segundo mejor indicador, validando que la agresividad emocional (gritos ortográficos) es clave para detectar haters.
   * Focalización (TIENE_ENTIDADES  0.025): Sigue siendo relevante, indicando que el odio tiende a señalar objetivos concretos.

2. Variables poco relevantes:
   * Estructura (NUM_ORACIONES): Con una correlación prácticamente nula (-0.001), se confirma que la complejidad estructural (número de frases) no sirve para discriminar; el odio puede ser tanto breve como extenso.
   * Ratio de Mayúsculas (RATIO_MAYUSCULAS): Con una correlación prácticamente nula (0.002), se confirma que el uso de mayúsculas no sirve para discriminar; el odio puede ser tanto en mayúsculas como minúsculas.

**Conclusión:** Se ha identificado que el odio tiene una "huella lingüística" de subjetividad e intensidad. Un modelo de Machine Learning aprovecharía estas variables (especialmente Adjetivos y Exclamaciones) para mejorar su precisión, combinándolas con el análisis semántico del texto.